In [1]:
import pandas as pd

In [2]:
import pandas as pd
df = pd.read_csv("train.csv")
print(df.columns.tolist())
df.head()

['project_id', 'num_commits', 'active_commit_days', 'project_span_days', 'commit_cadence', 'pct_commits_weekend', 'pct_commits_late_night', 'lines_added_total', 'lines_deleted_total', 'churn_ratio', 'median_lines_per_commit', 'max_lines_in_one_commit', 'pct_code_in_first_commit', 'num_commit_authors', 'avg_commit_msg_length', 'pct_generic_commit_msgs', 'num_branches', 'num_merge_commits', 'lines_of_code', 'num_source_files', 'num_directories', 'max_dir_depth', 'num_languages', 'primary_language', 'pct_primary_language', 'num_dependencies', 'framework', 'uses_game_engine', 'has_tests', 'num_test_files', 'has_ci_config', 'has_dockerfile', 'has_backend', 'has_database', 'avg_file_length', 'comment_line_ratio', 'num_image_files', 'num_audio_files', 'num_3d_model_files', 'num_font_files', 'asset_size_mb', 'has_custom_art', 'readme_word_count', 'readme_num_headings', 'readme_num_images', 'readme_num_code_blocks', 'readme_has_install_instructions', 'has_license', 'has_docs_folder', 'url_is_li

,project_id,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,...,forks,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years,hours_spent,log_hours,hours_bucket
0,535ce026f7,44,15,56,daily,0.2045,0.4545,8500,2900,0.3438,...,0,0,0,0,False,0,1-2,32.2,3.4726,30-100
1,869ba41ed7,9,3,76,sporadic,0.8889,0.0000,500,100,0.2484,...,0,0,0,0,False,0,3-5,5.0,1.6094,<10
2,d6f81f8862,12,1,0,frequent,1.0000,0.0000,5900,100,0.0247,...,0,0,0,0,False,0,3-5,1.3,0.2624,<10
3,152b5d915d,4,1,0,frequent,0.0000,0.2500,6000,0,0.0000,...,0,0,0,0,False,0,<1,2.0,0.6931,<10
4,3b5f8a49e6,7,3,64,sporadic,0.8571,0.0000,800,0,0.0419,...,0,0,0,0,False,0,1-2,4.0,1.3863,<10


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
train = pd.read_csv("train.csv")

DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
features = [c for c in train.columns if c not in DROP]
X = train[features].copy()
for col in features:
    if not pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].astype("category")

X_tr, X_va, y_tr, y_va, h_tr, h_va = train_test_split(
    X, train["log_hours"], train["hours_spent"], test_size=0.2, random_state=42
)
m = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, categorical_features="from_dtype", random_state=42)
m.fit(X_tr, y_tr)
preds = np.expm1(m.predict(X_va)).clip(0.1, 140.7)
print("Local RMSLE:", root_mean_squared_log_error(h_va, preds))

Local RMSLE: 0.7506365303506434


In [33]:
print("Columns:", train.columns.tolist())
print("\nData types:\n", train.dtypes)
print("\nFirst 3 rows:\n", train.head(3))

Columns: ['project_id', 'num_commits', 'active_commit_days', 'project_span_days', 'commit_cadence', 'pct_commits_weekend', 'pct_commits_late_night', 'lines_added_total', 'lines_deleted_total', 'churn_ratio', 'median_lines_per_commit', 'max_lines_in_one_commit', 'pct_code_in_first_commit', 'num_commit_authors', 'avg_commit_msg_length', 'pct_generic_commit_msgs', 'num_branches', 'num_merge_commits', 'lines_of_code', 'num_source_files', 'num_directories', 'max_dir_depth', 'num_languages', 'primary_language', 'pct_primary_language', 'num_dependencies', 'framework', 'uses_game_engine', 'has_tests', 'num_test_files', 'has_ci_config', 'has_dockerfile', 'has_backend', 'has_database', 'avg_file_length', 'comment_line_ratio', 'num_image_files', 'num_audio_files', 'num_3d_model_files', 'num_font_files', 'asset_size_mb', 'has_custom_art', 'readme_word_count', 'readme_num_headings', 'readme_num_images', 'readme_num_code_blocks', 'readme_has_install_instructions', 'has_license', 'has_docs_folder', '

In [36]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_log_error
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
train = pd.read_csv("train.csv")


def add_features(d):
    d = d.copy()
    d["commits_per_day"] = d.num_commits / (d.active_commit_days + 1)
    d["lines_per_commit"] = d.lines_added_total / (d.num_commits + 1)
    d["lines_per_active_day"] = d.lines_added_total / (d.active_commit_days + 1)
    d["loc_per_file"] = d.lines_of_code / (d.num_source_files + 1)
    d["log_lines_added"] = np.log1p(d.lines_added_total)
    d["log_commits"] = np.log1p(d.num_commits)
    d["span_per_active"] = d.project_span_days / (d.active_commit_days + 1)
    d["asset_total"] = (d.num_image_files + d.num_audio_files
                        + d.num_3d_model_files + d.num_font_files)
    d["churn_lines"] = d.lines_added_total + d.lines_deleted_total
    return d


train = add_features(train)
features = [c for c in train.columns if c not in DROP]

cat_cols = []
for c in features:
    if train[c].dtype == bool:
        train[c] = train[c].astype(int)
    elif not pd.api.types.is_numeric_dtype(train[c]):
        cat_cols.append(c)
        cats = sorted(train[c].dropna().astype(str).unique())
        train[c] = pd.Categorical(train[c].astype(str).where(train[c].notna()), categories=cats)

y = train["log_hours"].values
h_true = train["hours_spent"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_hgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_et = np.zeros(len(train))

X_ohe = pd.get_dummies(train[features], columns=cat_cols, dtype=float).fillna(-1)

for tr_idx, va_idx in kf.split(train):
    X_tr, y_tr = train.iloc[tr_idx][features], y[tr_idx]
    X_va, y_va = train.iloc[va_idx][features], y[va_idx]

    hgb = HistGradientBoostingRegressor(
        max_iter=900, learning_rate=0.015, max_leaf_nodes=6,
        min_samples_leaf=60, l2_regularization=10.0, max_features=0.5,
        categorical_features="from_dtype", random_state=42,
    )
    hgb.fit(X_tr, y_tr)
    oof_hgb[va_idx] = hgb.predict(X_va)

    if HAS_LGB:
        model_lgb = lgb.LGBMRegressor(
            n_estimators=1200, learning_rate=0.015, num_leaves=7,
            min_child_samples=55, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.5, reg_alpha=2.0, reg_lambda=10.0,
            random_state=42, verbose=-1,
        )
        model_lgb.fit(X_tr, y_tr)
        oof_lgb[va_idx] = model_lgb.predict(X_va)

    et = ExtraTreesRegressor(
        n_estimators=500, min_samples_leaf=5, max_features=0.5,
        n_jobs=-1, random_state=42,
    )
    et.fit(X_ohe.iloc[tr_idx], y_tr)
    oof_et[va_idx] = et.predict(X_ohe.iloc[va_idx])

print("HGB 5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(oof_hgb).clip(0.1, 140.7)), 6))
if HAS_LGB:
    print("LGB 5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(oof_lgb).clip(0.1, 140.7)), 6))
print("ET  5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(oof_et).clip(0.1, 140.7)), 6))

if HAS_LGB:
    blend = 0.35 * oof_hgb + 0.40 * oof_lgb + 0.25 * oof_et
else:
    blend = 0.50 * oof_hgb + 0.50 * oof_et

print("Blend 5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(blend).clip(0.1, 140.7)), 6))


HGB 5-Fold CV RMSLE: 0.723332
LGB 5-Fold CV RMSLE: 0.722494
ET  5-Fold CV RMSLE: 0.729642
Blend 5-Fold CV RMSLE: 0.715093


In [3]:
# Pre-ship check: runs the real predict.py on 500 rows it never trained on
import os, shutil, subprocess, sys, tempfile
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error

full = pd.read_csv("train.csv")
hold = full.sample(500, random_state=11)
rest = full.drop(hold.index)

tmp = tempfile.mkdtemp()
rest.to_csv(os.path.join(tmp, "train.csv"), index=False)
hold.drop(columns=["hours_spent", "log_hours", "hours_bucket"]).to_csv(os.path.join(tmp, "test.csv"), index=False)
shutil.copy("predict.py", tmp)

r = subprocess.run([sys.executable, "predict.py"], cwd=tmp, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("CRASHED:\n", r.stderr[-1500:])
else:
    sub = pd.read_csv(os.path.join(tmp, "predictions.csv"))
    assert len(sub) == len(hold) and sub.hours_spent.notna().all(), "bad predictions.csv"
    p = sub.set_index("project_id").loc[hold.project_id, "hours_spent"].values
    print("Rows:", len(sub), "| min/max hours:", sub.hours_spent.min(), sub.hours_spent.max())
    print("Holdout RMSLE (500 unseen rows):", round(root_mean_squared_log_error(hold.hours_spent.values, p), 4))
shutil.rmtree(tmp)

hgb: done (3 seeds, 3s)
lgb: done (3 seeds, 7s)
cat skipped (seed 42): No module named 'catboost'
et: done (3 seeds, 9s)
wrote predictions.csv with 500 rows

Rows: 500 | min/max hours: 1.06 37.9
Holdout RMSLE (500 unseen rows): 0.6267


In [2]:
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
import predict

train = pd.read_csv("train.csv")
y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values

X_tr, _, cat_cols = predict.prepare_data(train, train.head(5))
X_tr_xgb = X_tr.copy()
for c in cat_cols:
    X_tr_xgb[c] = X_tr_xgb[c].astype("category")

oof_cat = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_i, va_i) in enumerate(kf.split(X_tr)):
    cb = CatBoostRegressor(
        iterations=2000, learning_rate=0.025, depth=6, l2_leaf_reg=5.0,
        cat_features=cat_cols, random_seed=42 + fold, verbose=0, allow_writing_files=False,
    )
    cb.fit(X_tr.iloc[tr_i], y[tr_i], eval_set=(X_tr.iloc[va_i], y[va_i]), early_stopping_rounds=100)
    oof_cat[va_i] = cb.predict(X_tr.iloc[va_i])

    xm = XGBRegressor(
        n_estimators=2000, learning_rate=0.02, max_depth=4, min_child_weight=20,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=5.0,
        enable_categorical=True, tree_method="hist", early_stopping_rounds=100,
        random_state=42 + fold, n_jobs=-1, verbosity=0,
    )
    xm.fit(X_tr_xgb.iloc[tr_i], y[tr_i], eval_set=[(X_tr_xgb.iloc[va_i], y[va_i])], verbose=False)
    oof_xgb[va_i] = xm.predict(X_tr_xgb.iloc[va_i])

print("CatBoost 5-Fold CV:", round(root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(oof_cat))), 6))
print("XGBoost  5-Fold CV:", round(root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(oof_xgb))), 6))
print("Blend    5-Fold CV:", round(root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(0.75 * oof_cat + 0.25 * oof_xgb))), 6))

# Dry-run predict.py end-to-end
train.head(50).drop(columns=["hours_spent", "log_hours", "hours_bucket"]).to_csv("test.csv", index=False)
predict.main()
sub = pd.read_csv("predictions.csv")
print("Nulls:", sub.isna().sum().to_dict(), "| Sample:", sub["hours_spent"].head(5).round(2).tolist())
os.remove("test.csv")
os.remove("predictions.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'test.csv'